# Quiz Management System - Exploratory Data Analysis (EDA) & Analytics
**Project Domain:** Python for Data Science & Analytics  
**Certification Context:** E&ICT Academy, IIT Kanpur (*Quiz Management System, designed on Python for Data Science*)  
**Student:** Aryan Singh, University of Lucknow  
---
## 1. Project Overview & Business Problem
The objective of this study is to leverage educational assessment data collected by the **Quiz Management System** to perform:
1. **SQL-to-DataFrame Extraction & Data Preprocessing**
2. **Exploratory Data Analysis (EDA)** using **Pandas** & **NumPy**
3. **Statistical Inference & Engagement Correlation**
4. **Data Visualizations & Performance Dashboards** with **Matplotlib** & **Seaborn**
5. **Actionable Insights & Recommendations** for educators and academic coordinators

### 2. Environment Setup & Library Imports

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization settings
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
print('Data Science libraries imported successfully!')

### 3. Data Extraction from Relational SQLite Database
Extracting historical quiz attempts stored in `qms.db`.

In [ ]:
conn = sqlite3.connect('qms.db')
query = '''
SELECT 
    attempt_id,
    student_name,
    score,
    total_questions,
    score_percentage,
    time_taken_seconds,
    attempt_date,
    passed
FROM attempts
ORDER BY attempt_date DESC
'''
df = pd.read_sql_query(query, conn)
conn.close()

print(f'Total assessment records extracted: {len(df)}')
df.head()

### 4. Data Cleaning & Preprocessing
Checking for null values, verifying data types, and standardizing date columns.

In [ ]:
print('Missing Values Check:')
print(df.isnull().sum())

# Convert attempt_date to datetime
df['attempt_date'] = pd.to_datetime(df['attempt_date'])

# Add Performance Tier
df['Performance Tier'] = pd.cut(
    df['score_percentage'],
    bins=[-np.inf, 49.99, 74.99, 100],
    labels=['Needs Improvement (<50%)', 'Competent (50-74%)', 'Distinction (75-100%)']
)
print('\nData Types & Cleaned Preview:')
df.info()

### 5. Exploratory Data Analysis (EDA) & Summary Statistics

In [ ]:
# Summary statistics using Pandas
desc_stats = df[['score_percentage', 'time_taken_seconds']].describe()
print('Descriptive Statistics:')
display(desc_stats)

# Key Metrics using NumPy
scores = df['score_percentage'].values
print(f'Mean Score     : {np.mean(scores):.2f}%')
print(f'Median Score   : {np.median(scores):.2f}%')
print(f'Std Deviation  : {np.std(scores):.2f}%')
print(f'IQR (Q3 - Q1)  : {np.percentile(scores, 75) - np.percentile(scores, 25):.2f}%')
print(f'Overall Pass Rate: {(df["passed"].sum() / len(df)) * 100:.2f}%')

### 6. Correlation Analysis
Evaluating the statistical relationship between time taken, and final scoring.

In [ ]:
corr_matrix = df[['score_percentage', 'time_taken_seconds', 'passed']].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=1)
plt.title('Correlation Heatmap (Engagement Behavior vs. Performance)', fontsize=13, fontweight='bold')
plt.show()

### 7. Visual Analytics: Score Distribution & Engagement Scatter Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Score Distribution (KDE + Histogram)
sns.histplot(df['score_percentage'], kde=True, color='#1E40AF', bins=12, ax=axes[0])
axes[0].axvline(df['score_percentage'].mean(), color='red', linestyle='--', label=f'Mean: {df["score_percentage"].mean():.1f}%')
axes[0].axvline(df['score_percentage'].median(), color='green', linestyle='-.', label=f'Median: {df["score_percentage"].median():.1f}%')
axes[0].set_title('Learner Score Distribution (Histogram & KDE)', fontweight='bold')
axes[0].set_xlabel('Score Percentage (%)')
axes[0].legend()

# 2. Time Taken vs Score Percentage Scatter Plot with Regression Trend
sns.scatterplot(data=df, x='time_taken_seconds', y='score_percentage', hue='passed', palette={1: 'green', 0: 'red'}, s=60, alpha=0.85, ax=axes[1])
sns.regplot(data=df, x='time_taken_seconds', y='score_percentage', scatter=False, color='black', line_kws={'linestyle': '--', 'linewidth': 1.5}, ax=axes[1])
axes[1].set_title('Completion Time vs. Score Performance', fontweight='bold')
axes[1].set_xlabel('Time Taken (Seconds)')
axes[1].set_ylabel('Score Percentage (%)')
plt.tight_layout()
plt.show()

### 8. Performance Tiers Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].set_ylabel('Score Percentage (%)')

# 2. Donut Chart of Competency Tiers
tier_counts = df['Performance Tier'].value_counts()
axes[1].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%', startangle=140, colors=['#99ff99', '#66b3ff', '#ff9999'], wedgeprops=dict(width=0.4, edgecolor='white'))
axes[1].set_title('Competency Tier Breakdown', fontweight='bold')
plt.tight_layout()
plt.show()

### 9. Actionable Insights & Recommendations
1. **Cognitive Struggle Detection:** Strong negative correlation between time taken and score indicates that students requiring more time struggle with core foundations.
2. **Time Factor:** Extremely long completion times (>180s) correlate with failure; an optimal pacing threshold is between 60s and 120s.
3. **Curriculum Improvement:** Educators can review questions where time taken is disproportionately high to clarify instructional content.